# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **UCI Wine Dataset** dari UCI Machine Learning Repository. Dataset ini berisi hasil analisis kimia terhadap tiga jenis wine yang berasal dari kultivar berbeda di wilayah Italia yang sama. Tujuan pemodelan adalah melakukan klasifikasi `wine_class` berdasarkan 13 fitur numerik hasil pengukuran kimia.

Sumber dataset: https://archive.ics.uci.edu/dataset/109/wine

# **2. Import Library**

Pada tahap ini, library yang dibutuhkan untuk eksplorasi data, visualisasi, dan preprocessing akan diimpor terlebih dahulu.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

# **3. Memuat Dataset**

Dataset mentah dimuat dari folder `wine_raw` untuk memastikan proses experimentation menggunakan data raw sebelum dilakukan preprocessing.

In [ ]:
raw_path = Path("../wine_raw/wine.csv")
data = pd.read_csv(raw_path)

display(data.head())
print(f"Jumlah baris: {data.shape[0]}")
print(f"Jumlah kolom: {data.shape[1]}")
data.info()

# **4. Exploratory Data Analysis (EDA)**

Tahap EDA dilakukan untuk memahami struktur data, distribusi target, nilai hilang, duplikasi data, statistik deskriptif, dan hubungan antarfitur sebelum dataset diproses lebih lanjut.

In [ ]:
print("Jumlah missing value per kolom:")
display(data.isna().sum())

print(f"Jumlah duplikasi data: {data.duplicated().sum()}")

print("Distribusi target wine_class:")
display(data["wine_class"].value_counts().sort_index())

display(data.describe().T)

plt.figure(figsize=(6, 4))
sns.countplot(data=data, x="wine_class", hue="wine_class", palette="Set2", legend=False)
plt.title("Distribusi Kelas Wine")
plt.xlabel("Wine Class")
plt.ylabel("Jumlah Data")
plt.show()

plt.figure(figsize=(12, 8))
sns.heatmap(data.corr(numeric_only=True), cmap="coolwarm", center=0, linewidths=0.2)
plt.title("Korelasi Fitur Numerik")
plt.show()

# **5. Data Preprocessing**

Tahap preprocessing dilakukan dengan memisahkan kolom target, menangani missing value menggunakan median untuk fitur numerik, melakukan standardisasi fitur numerik menggunakan `StandardScaler`, lalu menyimpan dataset yang sudah siap dilatih ke folder `wine_preprocessing`.

In [ ]:
target_column = "wine_class"
processed_data = data.copy()
numeric_columns = [column for column in processed_data.columns if column != target_column]

missing_before = processed_data.isna().sum().to_dict()
for column in numeric_columns:
    processed_data[column] = processed_data[column].fillna(processed_data[column].median())

if processed_data[target_column].isna().any():
    processed_data[target_column] = processed_data[target_column].fillna(processed_data[target_column].mode()[0])

scaler = StandardScaler()
processed_data[numeric_columns] = scaler.fit_transform(processed_data[numeric_columns])
processed_data[target_column] = processed_data[target_column].astype(int)

output_dir = Path("wine_preprocessing")
output_dir.mkdir(parents=True, exist_ok=True)
processed_data.to_csv(output_dir / "processed_wine.csv", index=False)

summary = {
    "dataset": "UCI Wine Dataset",
    "source": "https://archive.ics.uci.edu/dataset/109/wine",
    "rows": int(processed_data.shape[0]),
    "columns": int(processed_data.shape[1]),
    "target_column": target_column,
    "target_distribution": processed_data[target_column].value_counts().sort_index().astype(int).to_dict(),
    "numeric_columns": numeric_columns,
    "missing_before": {key: int(value) for key, value in missing_before.items()},
    "missing_after": {key: int(value) for key, value in processed_data.isna().sum().to_dict().items()},
}

with (output_dir / "preprocessing_summary.json").open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

display(processed_data.head())
print(f"Dataset preprocessing tersimpan di: {output_dir / 'processed_wine.csv'}")